In [1]:
import pandas as pd
import os
from bs4 import BeautifulSoup
import pyarrow

In [2]:
def clean_filing(path):
    with open(path,"r",encoding="utf-8") as f:
        html=f.read()
    soup= BeautifulSoup(html,"lxml")
    tables= soup.find_all("table")
    for table in tables:
        table.decompose()
    text_after= soup.get_text(separator=" ")
    eliminate_point=text_after.lower().find("forward-looking statements")
    if eliminate_point==-1:
        body = text_after
        print("Warning: phase not found, kept full text")
    else:
        body=text_after[:eliminate_point]

    return body

In [4]:
records=[]
for ticker in os.listdir("data/raw"):
    print(ticker)
    ticket_folder=f"data/raw/{ticker}"
    for file_name in os.listdir(ticket_folder):
        file_path=f"{ticket_folder}\{file_name}"
        body=clean_filing(file_path)
        print(ticker, file_name, len(body))
        accession=file_name.replace(".htm","")
        records.append({"ticker":ticker,"accession":accession,"text":body})

filings=pd.DataFrame(records)
print(filings.head())
        

AAPL
AAPL 000032019317000006.htm 1933
AAPL 000032019317000067.htm 2125
AAPL 000032019318000005.htm 2153
AAPL 000032019318000067.htm 2801
AAPL 000032019318000098.htm 1867
AAPL 000032019318000142.htm 2994
AAPL 000032019319000002.htm 8655
AAPL 000032019319000007.htm 2926
AAPL 000032019319000063.htm 2842
AAPL 000032019319000073.htm 2401
AAPL 000032019319000117.htm 2421
AAPL 000032019320000008.htm 2520
AAPL 000032019320000050.htm 2999
AAPL 000032019320000060.htm 2866
AAPL 000032019320000094.htm 2704
AAPL 000032019321000009.htm 2559
AAPL 000032019321000055.htm 2822
AAPL 000032019321000063.htm 2372
AAPL 000032019321000104.htm 2532
AAPL 000032019322000006.htm 2341
AAPL 000032019322000058.htm 2531
AAPL 000032019322000069.htm 2368
AAPL 000032019322000107.htm 2796
AAPL 000032019323000005.htm 2265
AAPL 000032019323000063.htm 2381
AAPL 000032019323000075.htm 2212
AAPL 000032019323000104.htm 2202
AAPL 000032019324000005.htm 2497
AAPL 000032019324000067.htm 2555
AAPL 000032019324000080.htm 2408
AAPL 

In [5]:
filings["length"] = filings["text"].str.len()
filings.groupby("ticker")["length"].describe()

,count,mean,std,min,25%,50%,75%,max
ticker,,,,,,,,
AAPL,45.0,2604.955556,974.349829,1867.0,2230.00,2421.0,2763.00,8655.0
AMZN,24.0,21660.166667,10168.771007,6484.0,12848.00,22103.5,30053.75,35922.0
CF,91.0,18877.571429,12326.270603,71.0,2868.50,23423.0,28787.50,37516.0
COST,40.0,2214.300000,465.783773,1637.0,1867.25,2082.5,2493.00,3712.0
GOOGL,13.0,5059.307692,829.679897,3664.0,4546.00,4881.0,5434.00,7045.0
MSFT,24.0,6823.083333,1233.515299,4881.0,6081.25,6488.5,7630.75,9447.0
T,17.0,14401.411765,5953.814240,1207.0,12526.00,15556.0,17292.00,25680.0
TSLA,66.0,11612.772727,13040.409482,490.0,1872.50,3527.5,16220.25,47389.0
UNH,19.0,8554.315789,1928.063768,6889.0,7323.00,8144.0,8807.00,14378.0


In [6]:
filings

,ticker,accession,text,length
0,AAPL,000032019317000006,\n EX-99.1\n 2\n a8-kexhibit991q32017712017.ht...,1933
1,AAPL,000032019317000067,\n EX-99.1\n 2\n a8-kexhibit991q420179302017.h...,2125
2,AAPL,000032019318000005,\n EX-99.1\n 2\n a8-kexhibit991q12018123020.ht...,2153
3,AAPL,000032019318000067,\n EX-99.1\n 2\n a8-kexhibit991q220183312018.h...,2801
4,AAPL,000032019318000098,\n EX-99.1\n 2\n a8-kexhibit991q320186302018.h...,1867
...,...,...,...,...
334,UNH,000073176624000320,\n EX-99.1\n 2\n a2024q3exhibit991.htm\n EX-99...,8428
335,UNH,000073176625000022,\n EX-99.1\n 2\n a2024q4exhibit991.htm\n EX-99...,8266
336,UNH,000073176625000123,\n EX-99.1\n 2\n a2025q1exhibit991.htm\n EX-99...,7817
337,UNH,000073176625000228,\n EX-99.1\n 2\n a2025q2exhibit991.htm\n EX-99...,14378


In [7]:
os.makedirs("data/processed/", exist_ok=True)
filings.to_csv("data/processed/filings.csv",index=False)
